In [ ]:
print("hello")

In [ ]:
from pathlib import Path
from typing import Callable

import torch
import torch.nn.functional as F
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms

In [ ]:
dir_data = Path(".").resolve().with_name("data")
dir_train = dir_data / "train"

In [ ]:
len(list(dir_train.iterdir()))

In [ ]:
paths = list(dir_train.iterdir())

In [ ]:
path = list(dir_train.iterdir())[0]

In [ ]:
def get_label(path: Path) -> str:
    return path.stem.split(".")[0]

In [ ]:
get_label(path)

In [ ]:
path = paths[0]
img = Image.open(path).convert("RGB")
img

In [ ]:
transform = transforms.Compose(
    [
        transforms.Resize((32, 32)),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ]
)

In [ ]:
x = transform(img)
x.shape

In [ ]:
transforms.ToPILImage()(x)

In [ ]:
class CustomDataset(Dataset):
    encoding = {
        "dog": 0,
        "cat": 1,
    }
    decoding = list(encoding.values())

    def __init__(self, root_dir: Path | str, transform: Callable | None = None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.paths = list(self.root_dir.iterdir())

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx) -> tuple[torch.Tensor, int]:
        path = self.paths[idx]
        label = get_label(path)
        y = self.encoding[label]

        img = Image.open(path)
        x = self.transform(img)

        return x, y

In [ ]:
ds = CustomDataset(root_dir=dir_train, transform=transform)

In [ ]:
train_size = int(0.8 * len(ds))
val_size = len(ds) - train_size
train_size, val_size

In [ ]:
train_ds, valid_ds = random_split(ds, [train_size, val_size])
train_ds

In [ ]:
batch_size = 32

In [ ]:
train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
valid_dl = DataLoader(valid_ds, batch_size=batch_size, shuffle=False)
[o.shape for o in next(iter(train_dl))]

In [ ]:
isinstance(x, torch.Tensor)

In [ ]:
class ConvNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.fc1 = nn.Linear(8 * 8 * 64, 1000)
        self.fc2 = nn.Linear(1000, 10)

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.reshape(out.size(0), -1)
        out = self.fc1(out)
        out = self.fc2(out)
        return out

In [ ]:
model = ConvNet()

In [ ]:
x.shape
x[None,].shape

In [ ]:
model(x[None,]).shape

In [ ]:
model(x[None,])

In [ ]:
model(x[None,]).argmax()

In [ ]:
import lightning as L

In [ ]:
class LitConv(L.LightningModule):
    def __init__(self):
        super().__init__()
        self.model = ConvNet()

    def forward(self, x):
        out = self.model.forward(x)
        return out

    def _step(self, batch, batch_idx, set_: str):
        x, y = batch
        yhat = self.model.forward(x)
        loss = F.cross_entropy(yhat, y)
        self.log(f"{set_}_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def training_step(self, batch, batch_idx):
        return self._step(batch, batch_idx, "train")

    def validation_step(self, batch, batch_idx):
        return self._step(batch, batch_idx, "valid")

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)

In [ ]:
lit_conv = LitConv()
trainer = L.Trainer(max_epochs=5)
trainer.fit(lit_conv, train_dl)